# Bài 6.2 · Sentence embedding: biến câu thành vector rồi tìm theo nghĩa

**Câu hỏi của buổi này:** vector của một câu lấy ra bằng cách nào, đo hai câu "gần nhau" bằng gì, và model học từ cặp câu hơn model thô cùng cách đếm từ bao nhiêu khi so điểm và khi tìm kiếm?

Ba nguồn dữ liệu: **STS-B** (1 379 cặp câu tiếng Anh, mỗi cặp có điểm tương đồng do người chấm), **Quora** (câu hỏi trùng ý: mỗi truy vấn có đúng một câu hỏi tương ứng ẩn trong kho hàng nghìn câu) và **UIT-VSFC** (phản hồi sinh viên tiếng Việt, có nhãn chủ đề).

| Mục | Sẽ thấy |
|---|---|
| 1. Mean pooling có mask | tự viết, so với `encode`; quên mask làm vector lệch bao nhiêu |
| 2. Cosine giữa các câu | heatmap: câu cùng ý, câu trùng từ khác nghĩa |
| 3. STS-B | Spearman của TF-IDF, BERT/RoBERTa thô, các model câu; cosine của cặp ngẫu nhiên |
| 4. Truy hồi trên Quora | recall@k, MRR của BM25, TF-IDF và các embedding; vài truy vấn cụ thể |
| 5. Tiếng Việt và đa ngôn ngữ | câu Việt–Anh cùng nghĩa, model đa ngôn ngữ và model chỉ tiếng Anh |
| 6. Bản đồ chủ đề | t-SNE câu tiếng Việt tô theo chủ đề, phân loại chủ đề bằng embedding |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 5 phút. Bài giảng: [ai-canuc80k · bài 6.2](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/6.2).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time
import torch, transformers, sentence_transformers
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
transformers.logging.set_verbosity_error()

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_STS, N_QUERY, N_EXTRA, N_VI_TRAIN, N_VI_MAP = 1379, 1000, 9000, 3000, 1500
HF = {'bert': 'bert-base-uncased', 'roberta': 'roberta-base',
      'l6': 'sentence-transformers/all-MiniLM-L6-v2', 'multi': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
      'bge': 'BAAI/bge-small-en-v1.5', 'mpnet': 'sentence-transformers/all-mpnet-base-v2'}
NAME = {'bert': 'BERT thô (mean pool)', 'roberta': 'RoBERTa thô (mean pool)', 'l6': 'MiniLM-L6', 'multi': 'MiniLM đa ngôn ngữ',
        'bge': 'BGE-small', 'mpnet': 'MPNet-base'}
with quiet():
    M = {k: SentenceTransformer(v, device=device) for k, v in HF.items()}

def enc(key, texts):
    return M[key].encode(list(texts), batch_size=128, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)

def cut(s, n=62):
    return s if len(s) <= n else s[:n - 1] + '…'

table(pd.DataFrame([('torch', torch.__version__), ('sentence-transformers', sentence_transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Mean pooling phải biết token nào là thật

Encoder trả một vector cho mỗi token, shape `(B, L, d)`. Vector câu là trung bình các vector ấy, nhưng chỉ lấy token thật: câu ngắn trong batch bị **đệm** (padding) cho dài bằng câu dài nhất, và `attention_mask` ghi vị trí nào là thật. Dưới đây tự viết phép lấy trung bình có mask trên `all-MiniLM-L6-v2`, so với `SentenceTransformer.encode`, rồi cố tình quên mask.

In [ ]:
tok = AutoTokenizer.from_pretrained(HF['l6'])
with quiet():
    body = AutoModel.from_pretrained(HF['l6']).eval().to(device)
sents = ['Where can I borrow books?',
         'The cat sat on the mat.',
         'How do I reset my password if I no longer have access to my email?',
         'Rain is expected in the north of the country tomorrow afternoon.',
         'Thanks!',
         'The committee postponed the final decision on the new library opening hours until next month because several members were absent.']
batch = tok(sents, padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    H = body(**batch).last_hidden_state
m = batch['attention_mask'].unsqueeze(-1).to(H.dtype)
unit = lambda x: torch.nn.functional.normalize(x, dim=-1).cpu().numpy()
right = unit((H * m).sum(1) / m.sum(1).clamp_min(1))      # chỉ token thật
wrong = unit(H.mean(1))                                  # quên mask: tính cả vị trí đệm
ref = enc('l6', sents)
err = np.abs(right - ref).max()
check(f'mean pooling tự viết khớp SentenceTransformer.encode (sai số lớn nhất {err:.1e})', err < 1e-3)
check(f'H có shape (B, L, d) = {tuple(H.shape)}, vector câu có shape {right.shape}', H.shape[0] == len(sents) and right.shape == (len(sents), H.shape[2]))

real = batch['attention_mask'].sum(1).cpu().numpy(); L = H.shape[1]
cos_w = (right * wrong).sum(1)
P = pd.DataFrame({'câu': [cut(s, 58) for s in sents], 'token thật': real, 'vị trí đệm': L - real, 'cosine(quên mask, đúng)': cos_w})
table(P, f'Một batch {len(sents)} câu, độ dài đệm tới {L} token', fmt={'cosine(quên mask, đúng)': '{:.3f}'})

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(L - real, cos_w, s=70, color=C['clay'], zorder=3)
for i, (x, y) in enumerate(zip(L - real, cos_w)):
    ax.annotate(f'câu {i + 1}', (x, y), textcoords='offset points', xytext=(7, 7 if i % 2 == 0 else -13), fontsize=9)
ax.set(xlabel='số vị trí đệm của câu trong batch', ylabel='cosine với vector đúng', title='Quên mask: càng nhiều vị trí đệm, vector càng lệch')
plt.tight_layout(); plt.show()
long_i = int(np.argmax(real))
check('câu dài nhất (không có vị trí đệm) cho cùng vector dù có hay không mask', cos_w[long_i] > 0.9999)
check('cosine với vector đúng giảm khi số vị trí đệm tăng (tương quan âm)', np.corrcoef(L - real, cos_w)[0, 1] < 0)
short_i = int(np.argmin(real))
note(f'Câu ngắn nhất ({real[short_i]} token thật, {L - real[short_i]} vị trí đệm) có cosine chỉ <b>{cos_w[short_i]:.3f}</b> với vector đúng; câu dài nhất là {cos_w[long_i]:.3f}. '
     f'Cùng một câu nhưng vector phụ thuộc vào câu nào đứng chung batch: lỗi này không báo gì mà chỉ làm kết quả tìm kiếm dao động.')

## 2. Cosine giữa các câu

Vector đã chuẩn hóa L2 nên cosine bằng tích vô hướng: `E @ E.T` cho mọi cặp. Mười câu gồm bốn nhóm: hai câu cùng ý khác từ, một cặp mèo, hai câu chỉ đảo ngôi, và hai nghĩa của "bank".

In [ ]:
pairs = ['How do I reset my password?', 'I forgot my login credentials.',
         'What is the capital of France?', 'Which city is the capital of France?',
         'The cat sat on the mat.', 'A kitten is resting on the rug.',
         'The dog bites the man.', 'The man bites the dog.',
         'We sat on the river bank.', 'She deposited cash at the bank.']
E = enc('l6', pairs)
S = E @ E.T
fig, ax = plt.subplots(figsize=(9, 6.6))
im = ax.imshow(S, cmap='Blues', vmin=0, vmax=1); ax.grid(False)
for i in range(len(pairs)):
    for j in range(len(pairs)):
        ax.text(j, i, f'{S[i, j]:.2f}', ha='center', va='center', fontsize=8.5, color='white' if S[i, j] > .55 else '#222')
ax.set_xticks(range(len(pairs)), [str(i + 1) for i in range(len(pairs))]); ax.set_yticks(range(len(pairs)), [f'{i + 1}. {cut(p, 40)}' for i, p in enumerate(pairs)])
ax.set_title('Cosine giữa mười câu (MiniLM-L6)'); plt.colorbar(im, fraction=.04, pad=.02)
plt.tight_layout(); plt.show()
off = S[~np.eye(len(pairs), dtype=bool)]
check('đường chéo bằng 1 (câu so với chính nó)', np.allclose(np.diag(S), 1, atol=1e-4))
check('mọi cosine nằm trong [-1, 1]', S.min() >= -1.0001 and S.max() <= 1.0001)
check('câu cùng ý khác từ (1–2) gần nhau hơn câu không liên quan (1–3)', S[0, 1] > S[0, 2])
unrel = np.mean([S[i, j] for i in range(10) for j in range(10) if i // 2 != j // 2])
note(f'Cùng ý khác từ: reset mật khẩu/quên đăng nhập <b>{S[0, 1]:.2f}</b>, mèo/mèo con <b>{S[4, 5]:.2f}</b>. Hai nhóm khác chủ đề trung bình {unrel:.2f}. '
     f'Hai câu chỉ đảo ngôi "chó cắn người / người cắn chó" cho <b>{S[6, 7]:.2f}</b>, ' +
     ('cao ngang một cặp paraphrase: vector câu gần như không phân biệt ai cắn ai. ' if S[6, 7] > 0.9 else 'thấp hơn cặp paraphrase nhưng vẫn cao so với cặp khác chủ đề. ') +
     f'Hai nghĩa của "bank" được {S[8, 9]:.2f}: ' +
     (f'cao hơn hai câu khác chủ đề ({unrel:.2f}) vì trùng từ, nhưng thấp hơn hẳn cặp cùng ý.' if unrel < S[8, 9] < min(S[0, 1], S[4, 5]) else 'trùng một từ đã kéo hai câu khác nghĩa lại gần nhau đáng kể.'))

## 3. STS-B: model thô và model học từ cặp câu

STS-B cho 1 379 cặp câu kèm điểm tương đồng (đã chia về 0–1) do người chấm. Đo bằng **Spearman** giữa điểm người và cosine: chỉ quan tâm thứ tự, không quan tâm cosine lệch bao nhiêu. Baseline là TF-IDF; BERT và RoBERTa thô lấy trung bình hidden state **chưa học gì về cặp câu**; bốn model còn lại đã fine-tune trên dữ liệu cặp câu. Thêm một phép đo phụ: cosine trung bình của các cặp ghép ngẫu nhiên, cho biết các vector có tản ra hay dồn vào một vùng hẹp.

In [ ]:
with quiet():
    sts = pd.read_parquet('https://huggingface.co/api/datasets/sentence-transformers/stsb/parquet/default/test/0.parquet').head(N_STS)
a, b, gold = sts.sentence1.tolist(), sts.sentence2.tolist(), sts.score.to_numpy()
rp = np.random.default_rng(0).permutation(len(a))
rp = np.where(rp == np.arange(len(a)), (rp + 1) % len(a), rp)

tf = TfidfVectorizer(sublinear_tf=True).fit(a + b)
cos_tf = np.asarray(tf.transform(a).multiply(tf.transform(b)).sum(1)).ravel()
rows, COS = [('TF-IDF', 'đếm từ', spearmanr(gold, cos_tf)[0], np.nan)], {'tfidf': cos_tf}
for k in ['bert', 'roberta', 'l6', 'multi', 'bge', 'mpnet']:
    e1, e2 = enc(k, a), enc(k, b)
    COS[k] = (e1 * e2).sum(1)
    rows.append((NAME[k], 'model thô' if k in ('bert', 'roberta') else 'model câu', spearmanr(gold, COS[k])[0], (e1 * e2[rp]).sum(1).mean()))
R = pd.DataFrame(rows, columns=['phương pháp', 'loại', 'Spearman', 'cosine cặp ngẫu nhiên'])
table(R, f'STS-B test ({len(a)} cặp)', fmt={'Spearman': '{:.3f}', 'cosine cặp ngẫu nhiên': '{:.2f}'}, best='Spearman')

fig, ax = plt.subplots(figsize=(8, 3.8))
Rs = R.sort_values('Spearman')
colors = {'đếm từ': C['gray'], 'model thô': C['clay'], 'model câu': C['blue']}
ax.barh(Rs['phương pháp'], Rs['Spearman'], color=[colors[t] for t in Rs['loại']]); ax.grid(axis='y', visible=False)
for y, v in enumerate(Rs['Spearman']): ax.text(v + .01, y, f'{v:.3f}', va='center', fontsize=9)
ax.set(xlim=(0, 1.0), xlabel='Spearman với điểm người chấm', title='STS-B: xám đếm từ, cam model thô, xanh model câu')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, k, t in zip(axes, ['tfidf', 'bert', 'l6'], ['TF-IDF', NAME['bert'], NAME['l6']]):
    ax.scatter(gold, COS[k], s=6, alpha=.4, color=C['blue'] if k == 'l6' else C['clay'] if k == 'bert' else C['gray'])
    ax.set(xlabel='điểm người chấm (0–1)', title=f'{t}: Spearman {spearmanr(gold, COS[k])[0]:.2f}')
axes[0].set_ylabel('cosine')
plt.tight_layout(); plt.show()

best_k = max(['l6', 'multi', 'bge', 'mpnet'], key=lambda k: spearmanr(gold, COS[k])[0])
sp = {k: spearmanr(gold, v)[0] for k, v in COS.items()}
rnd = dict(zip(R['phương pháp'], R['cosine cặp ngẫu nhiên']))
check('model câu tốt nhất vượt BERT thô và RoBERTa thô về Spearman', sp[best_k] > max(sp['bert'], sp['roberta']))
check('mọi model câu vượt TF-IDF', all(sp[k] > sp['tfidf'] for k in ['l6', 'multi', 'bge', 'mpnet']))
note(f'Tốt nhất là <b>{NAME[best_k]}</b> với Spearman {sp[best_k]:.3f}; BERT thô chỉ {sp["bert"]:.3f}, ' +
     (f'thấp hơn cả TF-IDF ({sp["tfidf"]:.3f}): hidden state của BERT chưa được dạy rằng hai câu cùng ý phải gần nhau. ' if sp['bert'] < sp['tfidf'] else f'cao hơn TF-IDF ({sp["tfidf"]:.3f}) nhưng kém xa model câu. ') +
     f'Cosine của hai câu ngẫu nhiên: BERT thô <b>{rnd[NAME["bert"]]:.2f}</b>, RoBERTa thô <b>{rnd[NAME["roberta"]]:.2f}</b>, còn các model câu từ {min(rnd[NAME[k]] for k in ["l6", "multi", "bge", "mpnet"]):.2f} đến {max(rnd[NAME[k]] for k in ["l6", "multi", "bge", "mpnet"]):.2f}. '
     f'Vector của model thô dồn vào một vùng hẹp nên cosine ít phân biệt được hai câu.')

## 4. Truy hồi: tìm câu hỏi trùng ý trong kho

Quora cho cặp câu hỏi cùng ý. Lấy ngẫu nhiên các cặp: câu đầu làm **truy vấn**, câu thứ hai là **gold** và được bỏ vào kho cùng nhiều câu hỏi khác làm nhiễu. Mỗi truy vấn chỉ có một gold; câu nhiễu có thể cũng cùng ý mà không được gán nhãn, nên điểm dưới đây là cận dưới của chất lượng thật. Các phương pháp:

- **TF-IDF** cosine và **BM25** (k1 = 1,5; b = 0,75), cả hai tìm theo từ.
- Embedding: **BERT thô** và bốn model câu, chuẩn hóa L2 rồi `Q @ D.T`.

Hạng của gold là số tài liệu có điểm không thấp hơn nó, nên hòa tính theo hướng bất lợi. **Recall@k** là tỉ lệ truy vấn có gold trong top k; **MRR@10** là trung bình của 1/hạng (bằng 0 nếu gold ngoài top 10).

In [ ]:
with quiet():
    qq = pd.read_parquet('https://huggingface.co/api/datasets/sentence-transformers/quora-duplicates/parquet/pair/train/0.parquet')
qq = qq.drop_duplicates('anchor').drop_duplicates('positive').head(N_QUERY + N_EXTRA).reset_index(drop=True)
queries = qq.anchor[:N_QUERY].tolist()
corpus = qq.positive.tolist()           # gold của truy vấn i nằm ở vị trí i; phần sau là nhiễu
gold_i = np.arange(N_QUERY)

def bm25_scores(queries, corpus, k1=1.5, b=0.75):
    cv = CountVectorizer(lowercase=True)
    D = cv.fit_transform(corpus).tocsr().astype(np.float64)
    N = D.shape[0]; df_ = np.bincount(D.indices, minlength=D.shape[1])
    idf = np.log(1 + (N - df_ + .5) / (df_ + .5))
    dl = np.asarray(D.sum(1)).ravel()
    rows_ = np.repeat(np.arange(N), np.diff(D.indptr))
    W = D.copy()
    W.data = D.data * (k1 + 1) / (D.data + k1 * (1 - b + b * dl[rows_] / dl.mean())) * idf[D.indices]
    Q = cv.transform(queries).tocsr().astype(np.float64); Q.data[:] = 1
    return (Q @ W.T).toarray()

def ranks_of_gold(scores):
    g = scores[np.arange(len(scores)), gold_i]
    return (scores >= g[:, None]).sum(1)          # hòa tính bất lợi: gold đứng sau mọi tài liệu cùng điểm

SC = {}
tf2 = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)).fit(corpus)
SC['TF-IDF'] = (tf2.transform(queries) @ tf2.transform(corpus).T).toarray()
SC['BM25'] = bm25_scores(queries, corpus)
for k in ['bert', 'l6', 'multi', 'bge', 'mpnet']:
    SC[NAME[k]] = enc(k, queries) @ enc(k, corpus).T
RK = {n: ranks_of_gold(s) for n, s in SC.items()}
def metrics(r):
    return {'recall@1': (r <= 1).mean(), 'recall@5': (r <= 5).mean(), 'recall@10': (r <= 10).mean(), 'MRR@10': np.where(r <= 10, 1 / r, 0).mean()}
Q_ = pd.DataFrame({n: metrics(r) for n, r in RK.items()}).T.reset_index(names='phương pháp')
table(Q_, f'{N_QUERY} truy vấn, kho {len(corpus)} câu hỏi', fmt={c: '{:.3f}' for c in Q_.columns[1:]}, best='MRR@10')

ks = np.arange(1, 21)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1.3, 1]})
cols = {'TF-IDF': C['gray'], 'BM25': C['sand'], NAME['bert']: C['clay'], NAME['l6']: C['blue'], NAME['multi']: '#7a9cc6', NAME['bge']: C['green'], NAME['mpnet']: '#1f3f66'}
for n, r in RK.items():
    axes[0].plot(ks, [(r <= k).mean() for k in ks], label=n, color=cols[n], lw=2 if n in ('BM25', NAME['mpnet']) else 1.4)
axes[0].set(xlabel='k', ylabel='recall@k', title='Tỉ lệ truy vấn có gold trong top k'); axes[0].legend(fontsize=8.5, loc='lower right')
Qs = Q_.sort_values('MRR@10')
axes[1].barh(Qs['phương pháp'], Qs['MRR@10'], color=[cols[n] for n in Qs['phương pháp']]); axes[1].grid(axis='y', visible=False)
for y, v in enumerate(Qs['MRR@10']): axes[1].text(v + .01, y, f'{v:.3f}', va='center', fontsize=9)
axes[1].set(xlim=(0, 1.05), xlabel='MRR@10', title='MRR@10')
plt.tight_layout(); plt.show()

emb_names = [NAME[k] for k in ['l6', 'multi', 'bge', 'mpnet']]
mrr = Q_.set_index('phương pháp')['MRR@10']; r1 = Q_.set_index('phương pháp')['recall@1']
best_e = max(emb_names, key=lambda n: mrr[n])
check('recall@1 ≤ recall@5 ≤ recall@10 và MRR@10 ≥ recall@1 ở mọi phương pháp', ((Q_['recall@1'] <= Q_['recall@5']) & (Q_['recall@5'] <= Q_['recall@10']) & (Q_['MRR@10'] >= Q_['recall@1'])).all())
check('embedding tốt nhất vượt BM25 về MRR@10', mrr[best_e] > mrr['BM25'])
check('BERT thô kém hơn mọi model câu về MRR@10', all(mrr[NAME['bert']] < mrr[n] for n in emb_names))
note(f'MRR@10: BM25 <b>{mrr["BM25"]:.3f}</b>, TF-IDF {mrr["TF-IDF"]:.3f}, BERT thô {mrr[NAME["bert"]]:.3f}, tốt nhất <b>{best_e} {mrr[best_e]:.3f}</b> (recall@1 {r1[best_e]:.3f}, BM25 {r1["BM25"]:.3f}). ' +
     (f'BM25 tiếp tục là mốc mạnh: so với embedding tốt nhất chỉ kém {mrr[best_e] - mrr["BM25"]:.3f} MRR. ' if mrr[best_e] - mrr['BM25'] < 0.05 else f'Embedding hơn BM25 {mrr[best_e] - mrr["BM25"]:.3f} MRR khi hai câu hỏi diễn đạt khác nhau. ') +
     f'Điểm phụ thuộc kích thước và độ khó của kho, nên chỉ so các phương pháp trên cùng một kho.')

### Vài truy vấn cụ thể

Ba truy vấn mà BM25 để gold ngoài top 10 còn embedding tốt nhất đặt gold ở hạng 1, và hai truy vấn embedding đặt sai ở hạng 1. ✓ là gold, ✗ là câu khác.

In [ ]:
sc_e, rk_e, rk_b = SC[best_e], RK[best_e], RK['BM25']
win = np.flatnonzero((rk_b > 10) & (rk_e == 1))[:3]
lose = np.flatnonzero(rk_e > 1)[:2]
def row(i):
    top = np.argsort(-sc_e[i])[:3]
    return [cut(queries[i], 55)] + [('✓ ' if j == gold_i[i] else '✗ ') + cut(corpus[j], 55) for j in top] + [f'{rk_b[i]} → {rk_e[i]}']
table(pd.DataFrame([row(i) for i in list(win) + list(lose)], columns=['truy vấn', 'hạng 1', 'hạng 2', 'hạng 3', 'hạng gold: BM25 → embedding']),
      f'Top-3 của {best_e}: ba truy vấn thắng BM25, hai truy vấn sai')
n_win = int(((rk_b > 10) & (rk_e == 1)).sum()); n_lose = int(((rk_e > 10) & (rk_b == 1)).sum())
check('có truy vấn mà embedding thắng BM25 (gold ngoài top 10 → hạng 1)', n_win > 0)
note(f'Trên {N_QUERY} truy vấn, <b>{n_win}</b> truy vấn BM25 để gold ngoài top 10 mà embedding đặt ở hạng 1, ngược lại có {n_lose} truy vấn. '
     f'Hạng 1 "sai" không luôn là lỗi thật: kho có thể chứa một câu hỏi khác cũng cùng ý nhưng không được gán là gold.')

## 5. Tiếng Việt và đa ngôn ngữ

Tám cặp câu Việt–Anh cùng nghĩa. Với mỗi câu Việt, tìm câu Anh có cosine lớn nhất, trong tám câu. Model `all-MiniLM-L6` chỉ học trên tiếng Anh; `paraphrase-multilingual-MiniLM` học để câu cùng nghĩa ở nhiều thứ tiếng có vector gần nhau.

In [ ]:
vi = ['Tôi quên mật khẩu đăng nhập.', 'Thư viện mở cửa lúc mấy giờ?', 'Hôm nay trời mưa rất to.', 'Con mèo đang ngủ trên ghế.',
      'Giá vé tàu đi Đà Nẵng là bao nhiêu?', 'Tôi thích đọc sách vào buổi tối.', 'Giảng viên giải thích rất dễ hiểu.', 'Phòng học nóng quá.']
en = ['I forgot my login password.', 'What time does the library open?', 'It is raining heavily today.', 'The cat is sleeping on the chair.',
      'How much is a train ticket to Da Nang?', 'I like reading books in the evening.', 'The lecturer explains very clearly.', 'The classroom is too hot.']
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
top1, diag, offd = {}, {}, {}
for ax, k in zip(axes, ['l6', 'multi']):
    Sx = enc(k, vi) @ enc(k, en).T
    top1[k] = (Sx.argmax(1) == np.arange(8)).mean(); diag[k] = np.diag(Sx).mean(); offd[k] = Sx[~np.eye(8, dtype=bool)].mean()
    im = ax.imshow(Sx, cmap='Blues', vmin=0, vmax=1); ax.grid(False)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f'{Sx[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white' if Sx[i, j] > .55 else '#222')
    ax.set_xticks(range(8), [str(j + 1) for j in range(8)]); ax.set_yticks(range(8), [f'{i + 1}. {cut(v, 26)}' for i, v in enumerate(vi)])
    ax.set(xlabel='câu tiếng Anh (cùng số thứ tự là bản dịch đúng)', title=f'{NAME[k]}: đúng {int(top1[k] * 8)}/8')
plt.tight_layout(w_pad=6); plt.show()
check('model đa ngôn ngữ ghép đúng ít nhất 7/8 câu Việt với bản Anh', top1['multi'] >= 7 / 8)
check('model chỉ tiếng Anh ghép đúng ít hơn model đa ngôn ngữ', top1['l6'] < top1['multi'])
note(f'Model đa ngôn ngữ: cosine của cặp đúng trung bình <b>{diag["multi"]:.2f}</b>, của cặp sai {offd["multi"]:.2f}, ghép đúng {int(top1["multi"] * 8)}/8. '
     f'Model chỉ tiếng Anh: {diag["l6"]:.2f} và {offd["l6"]:.2f}, ghép đúng {int(top1["l6"] * 8)}/8 — ' +
     ('cặp dịch đúng gần như không nổi lên so với cặp sai.' if top1['l6'] <= 4 / 8 else 'vẫn khá hơn mong đợi, nhưng khoảng cách giữa cặp đúng và cặp sai hẹp hơn hẳn.'))

## 6. Bản đồ chủ đề tiếng Việt

UIT-VSFC gán mỗi phản hồi sinh viên một chủ đề: giảng viên, chương trình học, cơ sở vật chất, khác. Encode các câu bằng model đa ngôn ngữ, chiếu xuống 2 chiều bằng t-SNE và tô theo chủ đề. Để biết không gian này có chứa thông tin chủ đề hay không, huấn luyện hồi quy logistic trên vector (hoặc trên TF-IDF các âm tiết) rồi chấm macro-F1 trên tập test.

In [ ]:
URL = 'https://huggingface.co/api/datasets/uitnlp/vietnamese_students_feedback/parquet/default/{}/0.parquet'
with quiet():
    vtr = pd.read_parquet(URL.format('train')).sample(N_VI_TRAIN, random_state=0)
    vte = pd.read_parquet(URL.format('test'))
TOPICS = ['giảng viên', 'chương trình học', 'cơ sở vật chất', 'khác']
vte = vte.sample(min(N_VI_MAP, len(vte)), random_state=0)
f1s, EV = {}, {}
for name, k in [('MiniLM-L6 (chỉ tiếng Anh)', 'l6'), ('MiniLM đa ngôn ngữ', 'multi')]:
    EV[k] = enc(k, vte.sentence)
    clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(enc(k, vtr.sentence), vtr.topic)
    f1s[name] = f1_score(vte.topic, clf.predict(EV[k]), average='macro')
tv = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)
clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(tv.fit_transform(vtr.sentence), vtr.topic)
f1s = {'TF-IDF âm tiết': f1_score(vte.topic, clf.predict(tv.transform(vte.sentence)), average='macro'), **f1s}

xy = TSNE(2, perplexity=30, init='pca', random_state=0).fit_transform(EV['multi'])
tcol = [C['blue'], C['clay'], C['green'], C['gray']]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={'width_ratios': [1.5, 1]})
for t in range(4):
    sel = (vte.topic == t).to_numpy()
    axes[0].scatter(*xy[sel].T, s=9, alpha=.7, color=tcol[t], label=f'{TOPICS[t]} ({sel.sum()})')
axes[0].set(title=f't-SNE của {len(vte)} câu (MiniLM đa ngôn ngữ), tô theo chủ đề', xticks=[], yticks=[]); axes[0].grid(False); axes[0].legend(markerscale=2.5, loc='upper center', bbox_to_anchor=(.5, -.02), ncol=4)
axes[1].barh(list(f1s), list(f1s.values()), color=[C['gray'], C['clay'], C['blue']]); axes[1].grid(axis='y', visible=False)
for y, v in enumerate(f1s.values()): axes[1].text(v + .01, y, f'{v:.3f}', va='center', fontsize=9)
axes[1].set(xlim=(0, 1), xlabel='macro-F1 phân loại chủ đề (test)', title='Hồi quy logistic trên từng biểu diễn')
plt.tight_layout(); plt.show()
check('có đủ bốn chủ đề trong mẫu', vte.topic.nunique() == 4)
fk = list(f1s)
check('biểu diễn đa ngôn ngữ vượt chỉ tiếng Anh về macro-F1 chủ đề', f1s[fk[2]] > f1s[fk[1]])
d_tf = f1s[fk[2]] - f1s[fk[0]]
note(f'Macro-F1 chủ đề: TF-IDF âm tiết {f1s[fk[0]]:.3f}, MiniLM-L6 {f1s[fk[1]]:.3f}, đa ngôn ngữ <b>{f1s[fk[2]]:.3f}</b>. ' +
     ('Embedding đa ngôn ngữ hơn TF-IDF ngay cả khi chưa fine-tune. ' if d_tf > 0.01 else 'Embedding đa ngôn ngữ chỉ ngang TF-IDF: với chủ đề, từ khóa đã nói được gần hết. ' if d_tf > -0.01 else 'TF-IDF vẫn hơn embedding đa ngôn ngữ ở bài toán này. ') +
     f'Mẫu lệch: giảng viên chiếm {(vte.topic == 0).mean():.0%} số câu, nhóm "khác" chỉ {(vte.topic == 3).sum()} câu, nên macro-F1 mới là số đáng đọc; hình trái cho thấy các chủ đề chồng lấn ở ranh giới.')

## 7. Tóm tắt

In [ ]:
summary = [('mean pooling tự viết so với encode: sai số lớn nhất', f'{err:.1e}'),
           ('quên mask: cosine thấp nhất với vector đúng', f'{cos_w.min():.3f}'),
           ('STS-B Spearman: TF-IDF / BERT thô / ' + NAME[best_k], f"{sp['tfidf']:.3f} / {sp['bert']:.3f} / {sp[best_k]:.3f}"),
           ('cosine cặp ngẫu nhiên: BERT thô / MiniLM-L6', f"{rnd[NAME['bert']]:.2f} / {rnd[NAME['l6']]:.2f}"),
           ('Quora MRR@10: BM25 / ' + best_e, f'{mrr["BM25"]:.3f} / {mrr[best_e]:.3f}'),
           ('Quora recall@1: BM25 / ' + best_e, f'{r1["BM25"]:.3f} / {r1[best_e]:.3f}'),
           ('Việt–Anh ghép đúng: chỉ tiếng Anh / đa ngôn ngữ', f"{int(top1['l6'] * 8)}/8 / {int(top1['multi'] * 8)}/8"),
           ('macro-F1 chủ đề VSFC: TF-IDF / đa ngôn ngữ', f'{f1s[fk[0]]:.3f} / {f1s[fk[2]]:.3f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** tăng `N_EXTRA` lên 40 000 xem MRR của từng phương pháp giảm bao nhiêu; thêm câu của lớp vào danh sách `pairs` ở mục 2 rồi dự đoán trước ô nào sáng; thử tiền tố `'Represent this sentence for searching relevant passages: '` cho truy vấn khi dùng BGE; thay `normalize_embeddings=True` bằng `False` để thấy tích vô hướng thô khác cosine.